# 03 — Model Training

**AI-Based Retinal Imaging and Ophthalmic Screening System**

> 💡 **Recommended**: Run on **Google Colab with GPU** for fast training.
> Enable GPU: Runtime → Change runtime type → GPU (T4)

This notebook:
1. Loads and preprocesses the ODIR-5K dataset
2. Builds EfficientNetB0 with transfer learning
3. Trains Phase 1 (frozen backbone, classification head)
4. Fine-tunes Phase 2 (top backbone layers)
5. Saves the best model + training metadata
6. Plots training curves

In [ ]:
# ── Colab: install dependencies + upload project files ────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q datasets huggingface_hub tensorflow keras opencv-python scikit-learn matplotlib seaborn plotly
    
    # Option A: Mount Google Drive (if you have the project there)
    # from google.colab import drive
    # drive.mount('/content/drive')
    # sys.path.insert(0, '/content/drive/MyDrive/retinal_screening_ai')
    
    # Option B: Upload zip file
    # from google.colab import files
    # uploaded = files.upload()  # upload retinal_screening_ai.zip
    # !unzip -q retinal_screening_ai.zip
    
    sys.path.insert(0, '/content/retinal_screening_ai')
    
    # Create required directories
    !mkdir -p /content/retinal_screening_ai/models
    !mkdir -p /content/retinal_screening_ai/reports/figures
    !mkdir -p /content/retinal_screening_ai/reports/metrics
    !mkdir -p /content/retinal_screening_ai/data/splits

print('Setup complete.')

In [ ]:
# ── Local setup ───────────────────────────────────────────────────────────────
if not IN_COLAB:
    from pathlib import Path
    ROOT = Path('..').resolve()
    sys.path.insert(0, str(ROOT))

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

# Set seeds for reproducibility
from src.utils import set_seeds
set_seeds(42)
tf.random.set_seed(42)

In [ ]:
# ── Load and label dataset ────────────────────────────────────────────────────
from datasets import load_dataset
from src.dataset import build_dataframe, assign_four_class_labels
from src.preprocessing import run_full_pipeline

print('Loading ODIR-5K from HuggingFace ...')
ds = load_dataset('bumbledeep/odir', split='train')
df_meta = build_dataframe(ds)
df_labelled, stats = assign_four_class_labels(df_meta)
print(f'Labelled records: {len(df_labelled):,}')

In [ ]:
# ── Build numpy arrays + split ────────────────────────────────────────────────
(
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test,
    datagen_train
) = run_full_pipeline(ds, df_labelled)

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}    y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')

In [ ]:
# ── Class weights ─────────────────────────────────────────────────────────────
from src.utils import compute_class_weights
from config.config import TARGET_CLASSES

class_weights = compute_class_weights(y_train, num_classes=len(TARGET_CLASSES))
print('Class weights:', class_weights)

In [ ]:
# ── Build EfficientNetB0 model ────────────────────────────────────────────────
from src.train import build_model, compile_model
from config.config import LEARNING_RATE_HEAD, BATCH_SIZE, EPOCHS_HEAD

model = build_model(num_classes=len(TARGET_CLASSES), trainable_backbone=False)
model = compile_model(model, learning_rate=LEARNING_RATE_HEAD)
model.summary()

In [ ]:
# ── Phase 1: Train classification head ───────────────────────────────────────
from src.train import get_callbacks
from config.config import RANDOM_SEED

train_gen = datagen_train.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=RANDOM_SEED)

history_head = model.fit(
    train_gen,
    steps_per_epoch = max(1, len(X_train) // BATCH_SIZE),
    epochs          = EPOCHS_HEAD,
    validation_data = (X_val, y_val),
    callbacks       = get_callbacks('head'),
    class_weight    = class_weights,
    verbose         = 1,
)
print('Phase 1 training complete.')

In [ ]:
# ── Phase 2: Fine-tune top backbone layers ────────────────────────────────────
from config.config import (
    BEST_MODEL_PATH, FINETUNE_LAYERS,
    LEARNING_RATE_FINETUNE, EPOCHS_FINETUNE
)
from tensorflow import keras

if BEST_MODEL_PATH.exists():
    model.load_weights(str(BEST_MODEL_PATH))

# Unfreeze top N layers
backbone = model.get_layer(index=1)
for layer in backbone.layers[-FINETUNE_LAYERS:]:
    layer.trainable = True

model = compile_model(model, learning_rate=LEARNING_RATE_FINETUNE)

history_ft = model.fit(
    train_gen,
    steps_per_epoch = max(1, len(X_train) // BATCH_SIZE),
    epochs          = EPOCHS_FINETUNE,
    validation_data = (X_val, y_val),
    callbacks       = get_callbacks('finetune'),
    class_weight    = class_weights,
    verbose         = 1,
)
print('Phase 2 fine-tuning complete.')

In [ ]:
# ── Plot training history ─────────────────────────────────────────────────────
from src.train import plot_history
plot_history(history_head, history_ft)
print('Plots saved to reports/figures/')

In [ ]:
# ── Save class names and metadata ─────────────────────────────────────────────
from src.utils import save_class_names
from src.train import save_training_metadata
from config.config import CLASS_NAMES_PATH
import time

save_class_names(TARGET_CLASSES, CLASS_NAMES_PATH)
save_training_metadata(
    history_head, history_ft, class_weights,
    len(X_train), len(X_val), len(X_test),
    duration_seconds=0  # update manually if tracking time
)
print(f'Model saved to: {BEST_MODEL_PATH}')
print(f'Class names:    {CLASS_NAMES_PATH}')

In [ ]:
# ── Colab: Download model files ───────────────────────────────────────────────
if IN_COLAB:
    from google.colab import files
    print('Downloading model files ...')
    files.download(str(BEST_MODEL_PATH))
    files.download('/content/retinal_screening_ai/models/class_names.json')
    files.download('/content/retinal_screening_ai/models/training_metadata.json')
    print('Download initiated. Place files in the local models/ directory.')
else:
    print('Training complete. Model files saved locally in models/')

## Summary

- ✅ EfficientNetB0 with ImageNet weights
- ✅ Two-phase training (head → fine-tune)
- ✅ Class weights applied to handle imbalance
- ✅ EarlyStopping + ReduceLROnPlateau + ModelCheckpoint
- ✅ Best model saved to `models/best_model.keras`

**Next notebook:** `04_model_evaluation.ipynb`